# PatchTST Baseline — Channel-Independent (all 6 datasets)

Trains **canonical PatchTST** (channel-independent: each channel = separate sample, shared weights),
d_model=512, e_layers=3, patch_len=16, stride 8, lookback 512, pred 48.

Channel-independent avoids the O(C²) attention blowup that OOMs on 321/862-channel data.
This matches the original PatchTST paper and is what `benchmark_standard.py` evaluates.

**Datasets**: ETTh1, ETTh2, ETTm1, exchange_rate, electricity, traffic (full standard protocol)
**Runtime**: ~10–30 min total on T4
**Resumable**: re-run all cells → picks up from last checkpoint (`*_resume.pt`)
**Drive**: checkpoints to `MyDrive/nanoforecast-baselines/patchtst/`


## Step 1 — GPU check + Drive mount

In [ ]:
import torch, sys, os, json, time, subprocess
assert torch.cuda.is_available(), 'No GPU. Runtime → Change runtime type → T4 GPU.'
print(f'PyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name(0)}')

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/nanoforecast-baselines/patchtst'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive ready:', DRIVE_ROOT)


## Step 2 — Fetch benchmark code from GitHub (no upload needed)

The trainer + vendored PatchTST + harness were pushed to the repo (branch `v0.5`),
so we shallow-clone and import straight from the clone. No tarball to upload.

In [ ]:
# Shallow clone of the repo (benchmarks/ + benchmark_standard.py + nanoforecast/)
# Idempotent: always re-clone fresh so the latest code (incl. the 40-epoch
# training schedule) is used even if the cell was run before in this session.
!rm -rf /content/NanoForecast && git clone --depth 1 -b v0.5 https://github.com/eulogik/NanoForecast /content/NanoForecast

import os, sys
sys.path.insert(0, '/content/NanoForecast')
os.chdir('/content/NanoForecast')

need = ['benchmark_standard.py',
        'benchmarks/train_patchtst_ci.py',
        'benchmarks/tsl/PatchTST.py',
        'benchmarks/tsl/Embed.py']
missing = [f for f in need if not os.path.exists(f)]
assert not missing, f'Missing files: {missing}'
print('Files OK:')
!ls benchmark_standard.py benchmarks/ benchmarks/tsl/


## Step 3 — Train all 6 datasets (channel-independent)

Run the cell below. Each dataset trains until early-stop (patience 3); checkpoints
and `*.json` metadata land in `MyDrive/nanoforecast-baselines/patchtst/` (via symlink).
Re-running resumes: datasets with a final `{ds}.pt` are skipped, partial runs resume.

In [ ]:
# Train with the LOCAL repo checkpoint dir. NO Drive symlink: Google Drive
# fuse handles symlinks unreliably and torch.save died with
# 'Parent directory does not exist' mid-training. Drive is a plain backup.
ckpt_dir = '/content/NanoForecast/benchmarks/checkpoints/patchtst'
if os.path.islink(ckpt_dir):
    os.unlink(ckpt_dir)

# RETRAIN FRESH: remove stale local checkpoints/resumes so the new 40-epoch
# schedule applies from scratch (previous run stopped at 4-6 epochs).
import glob
removed = 0
for f in glob.glob(os.path.join(ckpt_dir, '*.pt')) + glob.glob(os.path.join(ckpt_dir, '*.json')):
    os.remove(f)
    removed += 1
print(f'removed {removed} stale checkpoint file(s); training dir: {ckpt_dir}')

from benchmarks.train_patchtst_ci import train_one
import shutil

DATASETS = ['ETTh1', 'ETTh2', 'ETTm1', 'exchange_rate', 'electricity', 'traffic']
for ds in DATASETS:
    print(f'\n=== Training {ds} ===')
    t0 = time.time()
    r = train_one(ds, 'cuda')
    if r:
        print(f'  {ds}: best_val_mse={r["best_val_mse"]:.6f} '
              f'epochs={r["epochs"]} ({time.time()-t0:.0f}s)')
    else:
        print(f'  {ds}: skipped (checkpoint exists)')
    for ext in ('pt', 'json'):
        src = os.path.join(ckpt_dir, f'{ds}.{ext}')
        if os.path.exists(src):
            shutil.copy(src, os.path.join(DRIVE_ROOT, f'{ds}.{ext}'))
    torch.cuda.empty_cache()

print('\n=== ALL DONE ===')
print('Local:', sorted(os.listdir(ckpt_dir)))
print('Drive backup:', sorted(os.listdir(DRIVE_ROOT)))


## Step 4 — Benchmark all 6 datasets on this GPU

Runs the standard protocol directly on Colab (T4) — no local download needed.
Results land in `/content/NanoForecast/results/standard_benchmark.json`.

In [ ]:
import os
os.environ['NF_DEVICE'] = 'cuda'  # benchmark_standard uses this (not mps/cpu)

!cd /content/NanoForecast && python3 benchmark_standard.py --models patchtst --datasets ETTh1,ETTh2,ETTm1,exchange_rate,electricity,traffic

import json
res = json.load(open('/content/NanoForecast/results/standard_benchmark.json'))
pt = res['results'].get('patchtst', {})
print('\n=== PATCHTST MASE (standard protocol) ===')
for ds, m in pt.items():
    print(f'  {ds:15s} {m["mase"]:.4f}')


## Step 5 — Push checkpoints + results to Hugging Face (optional)

Uploads the 6 `.pt`/`.json` pairs and the benchmark results to
[`eulogik/nanoforecast-patchtst-baselines`](https://huggingface.co/new).
Local runs then auto-fetch weights via `benchmark_standard.py` (no manual download).
Run `notebook_login()` if you haven't set a token (needs *write* access).

In [ ]:
from huggingface_hub import notebook_login, HfApi
notebook_login()  # only needed once per session

REPO = "eulogik/nanoforecast-patchtst-baselines"
api = HfApi()
api.create_repo(REPO, repo_type="model", exist_ok=True)

readme = '''---
tags: [time-series-forecasting, patchtst, benchmark]
---

# PatchTST baselines (channel-independent, standard protocol)

Trained on a free Colab T4 with the exact hyperparameters of the PatchTST paper
(d_model=512, e_layers=3, patch_len=16, stride 8, dropout 0.3, lr 1e-4, patience 3,
lookback 512, horizon 48). Channel-independent: each channel is a separate sample.

Contains `{ds}.pt` + `{ds}.json` for ETTh1, ETTh2, ETTm1, exchange_rate, electricity,
traffic, plus `standard_benchmark.json` (MASE/MAE/MSE/sMAPE under the protocol in
`benchmark_standard.py` of the NanoForecast repo).
'''
api.upload_file(path_or_fileobj=readme.encode(),
                path_in_repo='README.md', repo_id=REPO)

for f in sorted(os.listdir(DRIVE_ROOT)):
    if f.endswith(('.pt', '.json')):
        api.upload_file(path_or_fileobj=os.path.join(DRIVE_ROOT, f),
                        path_in_repo=f, repo_id=REPO)
        print('uploaded', f)

api.upload_file(
    path_or_fileobj='/content/NanoForecast/results/standard_benchmark.json',
    path_in_repo='standard_benchmark.json', repo_id=REPO)
print('\nDone -> https://huggingface.co/' + REPO)


## Step 6 — Verify on Drive

In [ ]:
for f in sorted(os.listdir(DRIVE_ROOT)):
    sz = os.path.getsize(os.path.join(DRIVE_ROOT, f))
    print(f'  {f:30s} {sz/1024:8.1f} KB')
